<a href="https://colab.research.google.com/github/fladroid/balsam/blob/main/01_data_prep.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# Cell 1 - Instalacija
!pip install paramiko psycopg2-binary -q


In [ ]:
# Cell 2 - Imports i mount
from google.colab import drive
drive.mount('/content/drive')

import pandas as pd
import requests
import os

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
# Cell 3 - Skini Tatoeba sentences
!wget -q https://downloads.tatoeba.org/exports/sentences.tar.bz2 -O /content/sentences.tar.bz2
!tar -xjf /content/sentences.tar.bz2 -C /content/
!ls -lh /content/sentences.csv

-rw-r--r-- 1 109 115 707M Feb 28 06:01 /content/sentences.csv


In [ ]:
# Cell 4 - Pogledaj strukturu
df = pd.read_csv('/content/sentences.csv',
                 sep='\t',
                 header=None,
                 names=['id', 'lang', 'text'])
print(df.shape)
print(df['lang'].value_counts().head(20))

(13291595, 3)
lang
eng    2013074
rus    1187058
ita     958520
epo     809128
kab     782870
deu     758446
tur     744909
fra     707296
ber     693138
por     439903
spa     435281
hun     429436
jpn     247954
heb     211493
nld     198077
ukr     187876
fin     154212
lit     150419
pol     135249
ces      86401
Name: count, dtype: int64


In [ ]:
# Cell 5 - Filtriraj naše jezike
# srp = srpski, hrv = hrvatski, bos = bosanski
our_langs = ['srp', 'hrv', 'bos', 'eng']
df_our = df[df['lang'].isin(our_langs)]
print(df_our['lang'].value_counts())

lang
eng    2013074
srp      51352
hrv       6141
bos       1741
Name: count, dtype: int64


In [ ]:
# Cell 6 - Skini links fajl NA DRIVE
import os

data_path = '/content/drive/MyDrive/balsam/data/'
os.makedirs(data_path, exist_ok=True)

# Skini samo ako već ne postoji
if not os.path.exists(data_path + 'links.csv'):
    !wget -q https://downloads.tatoeba.org/exports/links.tar.bz2 -O {data_path}links.tar.bz2
    !tar -xjf {data_path}links.tar.bz2 -C {data_path}
    print("Skinuto i raspakovano!")
else:
    print("Već postoji, preskačem download!")

Već postoji, preskačem download!


In [ ]:
# Sačuvaj sentences na Drive ako već nije tamo
if not os.path.exists(data_path + 'sentences.csv'):
    !cp /content/sentences.csv {data_path}sentences.csv
    print("Sentences sačuvan na Drive!")
else:
    print("Već postoji!")

Već postoji!


In [ ]:
# Cell 7 - Ucitaj sve sa Drive i napravi parove
data_path = '/content/drive/MyDrive/balsam/data/'

# Ucitaj sentences sa Drive
df = pd.read_csv(data_path + 'sentences.csv',
                 sep='\t',
                 header=None,
                 names=['id', 'lang', 'text'])
print(f"Sentences: {len(df)}")

# Ucitaj links sa Drive
links = pd.read_csv(data_path + 'links.csv',
                    sep='\t',
                    header=None,
                    names=['id1', 'id2'])
print(f"Links: {len(links)}")

# Index po id
df_indexed = df.set_index('id')

# Funkcija za parove
def get_pairs(lang1, lang2, df_indexed, links):
    ids_lang1 = set(df_indexed[df_indexed['lang'] == lang1].index)
    ids_lang2 = set(df_indexed[df_indexed['lang'] == lang2].index)

    mask = (links['id1'].isin(ids_lang1) & links['id2'].isin(ids_lang2)) | \
           (links['id1'].isin(ids_lang2) & links['id2'].isin(ids_lang1))

    return links[mask].copy()

for lang in ['srp', 'hrv', 'bos']:
    pairs = get_pairs(lang, 'eng', df_indexed, links)
    print(f"{lang}-eng parovi: {len(pairs)}")

Sentences: 13291595
Links: 27758924
srp-eng parovi: 66702
hrv-eng parovi: 6164
bos-eng parovi: 1248


In [ ]:
# Cell 8 - Izgradi parove sa tekstom
def build_pair_df(lang, df_indexed, links):
    ids_lang = set(df_indexed[df_indexed['lang'] == lang].index)
    ids_eng = set(df_indexed[df_indexed['lang'] == 'eng'].index)

    mask = (links['id1'].isin(ids_lang) & links['id2'].isin(ids_eng)) | \
           (links['id1'].isin(ids_eng) & links['id2'].isin(ids_lang))

    pairs = links[mask].copy()

    result = []
    for _, row in pairs.iterrows():
        id1, id2 = row['id1'], row['id2']
        t1 = df_indexed.loc[id1]
        t2 = df_indexed.loc[id2]

        if t1['lang'] == lang:
            result.append({'lang': lang, 'local_text': t1['text'], 'eng_text': t2['text']})
        else:
            result.append({'lang': lang, 'local_text': t2['text'], 'eng_text': t1['text']})

    return pd.DataFrame(result)

dfs = {}
for lang in ['srp', 'hrv', 'bos']:
    print(f"Gradim {lang}...")
    dfs[lang] = build_pair_df(lang, df_indexed, links)
    print(f"{lang}: {len(dfs[lang])} parova")
    print(dfs[lang].head(2))
    print()

Gradim srp...
srp: 66702 parova
  lang                local_text                eng_text
0  srp   Hajde da probamo nešto.    Let's try something.
1  srp  Морам да идем да спавам.  I have to go to sleep.

Gradim hrv...
hrv: 6164 parova
  lang             local_text                    eng_text
0  hrv     Moram ići spavati.      I have to go to sleep.
1  hrv  Zaporka je "Muiriel".  The password is "Muiriel".

Gradim bos...
bos: 1248 parova
  lang local_text          eng_text
0  bos    Požuri.         Hurry up.
1  bos  Čestitam!  Congratulations!



In [ ]:
# Cell 9 - Sačuvaj CSV na Drive
for lang in ['srp', 'hrv', 'bos']:
    path = f"{data_path}{lang}_eng_pairs.csv"
    dfs[lang].to_csv(path, index=False)
    print(f"Sačuvano: {path}")

Sačuvano: /content/drive/MyDrive/balsam/data/srp_eng_pairs.csv
Sačuvano: /content/drive/MyDrive/balsam/data/hrv_eng_pairs.csv
Sačuvano: /content/drive/MyDrive/balsam/data/bos_eng_pairs.csv


In [ ]:
# Cell 10 - Konekcija + insert (kompletna, robusna verzija)
import paramiko, psycopg2, threading, time, os, pandas as pd
from getpass import getpass
import socket

data_path = '/content/drive/MyDrive/balsam/data/'
key_path = '/content/drive/MyDrive/balsam/balsam_key.txt'
os.chmod(key_path, 0o600)

db_password = getpass("Unesi DB password: ")

# Oslobodi port 5433 ako je zauzet
def free_port(port):
    try:
        s = socket.socket(socket.AF_INET, socket.SOCK_STREAM)
        s.setsockopt(socket.SOL_SOCKET, socket.SO_REUSEADDR, 1)
        s.bind(('localhost', port))
        s.close()
    except:
        pass

free_port(5433)

# SSH konekcija
ssh = paramiko.SSHClient()
ssh.set_missing_host_key_policy(paramiko.AutoAddPolicy())
ssh.connect('balsam.dynu.net', username='fla', key_filename=key_path)
print("SSH OK!")

transport = ssh.get_transport()

def forward_tunnel(local_port, remote_host, remote_port, transport):
    server = socket.socket(socket.AF_INET, socket.SOCK_STREAM)
    server.setsockopt(socket.SOL_SOCKET, socket.SO_REUSEADDR, 1)
    server.bind(('localhost', local_port))
    server.listen(5)
    while True:
        try:
            client_conn, addr = server.accept()
            chan = transport.open_channel('direct-tcpip', (remote_host, remote_port), addr)
            def handle(c, ch):
                while True:
                    try:
                        data = c.recv(4096)
                        if not data: break
                        ch.send(data)
                        data = ch.recv(4096)
                        if not data: break
                        c.send(data)
                    except:
                        break
                c.close()
                ch.close()
            threading.Thread(target=handle, args=(client_conn, chan), daemon=True).start()
        except:
            break

t = threading.Thread(target=forward_tunnel, args=(5433, 'localhost', 5432, transport), daemon=True)
t.start()
time.sleep(2)

# DB konekcija
conn = psycopg2.connect(host='localhost', port=5433, database='balsam', user='pgu', password=db_password)
cur = conn.cursor()
print("DB OK!")

# Kreiraj tabelu
cur.execute("DROP TABLE IF EXISTS sentence_pairs;")
cur.execute("""
    CREATE TABLE sentence_pairs (
        lang VARCHAR(3) NOT NULL,
        local_text TEXT NOT NULL,
        eng_text TEXT NOT NULL
    );
""")
conn.commit()
print("Tabela kreirana!")

# Insert svih podataka u batch-evima
for lang in ['srp', 'hrv', 'bos']:
    df_lang = pd.read_csv(f"{data_path}{lang}_eng_pairs.csv")
    rows = [tuple(r) for r in df_lang[['lang','local_text','eng_text']].values]

    batch_size = 500
    total_batches = (len(rows) + batch_size - 1) // batch_size

    for i in range(0, len(rows), batch_size):
        batch = rows[i:i+batch_size]
        cur.executemany(
            "INSERT INTO sentence_pairs (lang, local_text, eng_text) VALUES (%s, %s, %s)",
            batch
        )
        conn.commit()
        print(f"{lang} - {i//batch_size + 1}/{total_batches} OK")

    print(f"✓ {lang} završen: {len(rows)} redova")

conn.close()
ssh.close()
print("\nSve gotovo!")

Unesi DB password: ··········
SSH OK!
DB OK!
Tabela kreirana!
srp - 1/134 OK
srp - 2/134 OK
srp - 3/134 OK
srp - 4/134 OK
srp - 5/134 OK
srp - 6/134 OK
srp - 7/134 OK
srp - 8/134 OK
srp - 9/134 OK
srp - 10/134 OK
srp - 11/134 OK
srp - 12/134 OK
srp - 13/134 OK
srp - 14/134 OK
srp - 15/134 OK
srp - 16/134 OK
srp - 17/134 OK
srp - 18/134 OK
srp - 19/134 OK
srp - 20/134 OK
srp - 21/134 OK
srp - 22/134 OK
srp - 23/134 OK
srp - 24/134 OK
srp - 25/134 OK
srp - 26/134 OK
srp - 27/134 OK
srp - 28/134 OK
srp - 29/134 OK
srp - 30/134 OK
srp - 31/134 OK
srp - 32/134 OK
srp - 33/134 OK
srp - 34/134 OK
srp - 35/134 OK
srp - 36/134 OK
srp - 37/134 OK
srp - 38/134 OK
srp - 39/134 OK
srp - 40/134 OK
srp - 41/134 OK
srp - 42/134 OK
srp - 43/134 OK
srp - 44/134 OK
srp - 45/134 OK
srp - 46/134 OK
srp - 47/134 OK
srp - 48/134 OK
srp - 49/134 OK
srp - 50/134 OK
srp - 51/134 OK
srp - 52/134 OK
srp - 53/134 OK
srp - 54/134 OK
srp - 55/134 OK
srp - 56/134 OK
srp - 57/134 OK
srp - 58/134 OK
srp - 59/134 OK
srp